## Data Cleaning

#### Since dataset contains unamed:0

In [2]:
import pandas as pd
import numpy as np

df=pd.read_csv("../audited_data.csv")

df.head()

,Unnamed: 0,building_id,meter,timestamp,meter_reading,site_id,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed,primary_use,square_feet,year_built,floor_count
0,16777087,1020,0,2016-11-02 00:00:00,86.6301,10,8.9,4.0,3.3,0.0,1016.5,350.0,3.6,Entertainment/public assembly,22101,NaN,2.0
1,15143945,987,2,2016-10-04 00:00:00,228.7500,9,27.8,0.0,15.6,0.0,1010.0,110.0,2.6,Education,166395,NaN,NaN
2,903842,113,0,2016-01-17 19:00:00,335.6250,1,3.6,NaN,1.5,NaN,1023.0,120.0,4.6,Education,100481,1958.0,9.0
3,10190603,293,0,2016-07-07 05:00:00,777.2200,3,27.2,0.0,24.4,0.0,1011.0,170.0,3.1,Office,408000,2010.0,NaN
4,9600591,1176,2,2016-06-26 15:00:00,238.2810,13,25.0,2.0,10.0,0.0,1014.5,250.0,5.1,Education,142672,NaN,NaN


In [3]:
df.drop(columns=['Unnamed: 0'], inplace=True)

In [ ]:
df.columns

Index(['building_id', 'meter', 'timestamp', 'meter_reading', 'site_id',
       'air_temperature', 'cloud_coverage', 'dew_temperature',
       'precip_depth_1_hr', 'sea_level_pressure', 'wind_direction',
       'wind_speed', 'primary_use', 'square_feet', 'year_built',
       'floor_count'],
      dtype='str')

## Convert Timestamp

In [5]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 409 entries, 0 to 408
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   building_id         409 non-null    int64         
 1   meter               409 non-null    int64         
 2   timestamp           409 non-null    datetime64[us]
 3   meter_reading       409 non-null    float64       
 4   site_id             409 non-null    int64         
 5   air_temperature     408 non-null    float64       
 6   cloud_coverage      225 non-null    float64       
 7   dew_temperature     408 non-null    float64       
 8   precip_depth_1_hr   327 non-null    float64       
 9   sea_level_pressure  385 non-null    float64       
 10  wind_direction      382 non-null    float64       
 11  wind_speed          407 non-null    float64       
 12  primary_use         409 non-null    str           
 13  square_feet         409 non-null    int64         
 14  year_

## Create Missing Value Report

In [7]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': round(df.isnull().mean()*100,2)
})

missing.sort_values(
    by='Missing %',
    ascending=False
)

,Missing Count,Missing %
floor_count,330,80.68
year_built,245,59.90
cloud_coverage,184,44.99
precip_depth_1_hr,82,20.05
wind_direction,27,6.60
sea_level_pressure,24,5.87
wind_speed,2,0.49
air_temperature,1,0.24
dew_temperature,1,0.24
building_id,0,0.00


## Save Report 

In [10]:
missing.to_csv(
    '../reports/missing_values_report.csv'
)

### Drop floor Count

#### Reason : 80% of value missing

In [11]:
df.drop(
    columns=['floor_count'],
    inplace=True
)

### Fill Missing Weather Values

In [12]:
weather_cols = [
    'air_temperature',
    'cloud_coverage',
    'dew_temperature',
    'precip_depth_1_hr',
    'sea_level_pressure',
    'wind_direction',
    'wind_speed'
]

for col in weather_cols:
    df[col] = df[col].fillna(
        df[col].median()
    )

### Verification 

In [13]:
df.isnull().sum()

building_id             0
meter                   0
timestamp               0
meter_reading           0
site_id                 0
air_temperature         0
cloud_coverage          0
dew_temperature         0
precip_depth_1_hr       0
sea_level_pressure      0
wind_direction          0
wind_speed              0
primary_use             0
square_feet             0
year_built            245
dtype: int64

### Business Feature Enginneering

#### Now we create fields an energy consultant actually uses.

### Extract Month

In [14]:
df['month'] = df['timestamp'].dt.month

### Extract Quarter

In [15]:
df['quarter'] = df['timestamp'].dt.quarter

### Extract Day

In [16]:
df['day'] = df['timestamp'].dt.day

### NB: These become Power BI filters.

## Energy KPI Creation

### Total Energy Consumption 

In [17]:
total_energy = df['meter_reading'].sum()

print(total_energy)

147395.8958


### Average Energy Consumption

In [18]:
avg_energy = df['meter_reading'].mean()

print(avg_energy)

360.38116332518337


## Energy Intensity

In [19]:
df['energy_intensity'] = (
    df['meter_reading']
    /
    df['square_feet']
)

##### This measures: Energy used per square foot. A standard energy efficiency metric.

## Carbon Emissions

In [20]:
EMISSION_FACTOR = 0.43

In [21]:
df['co2_emissions'] = (
    df['meter_reading']
    *
    EMISSION_FACTOR
)

In [22]:
total_emissions = (
    df['co2_emissions']
    .sum()
)

print(total_emissions)

63380.23519399999


## Business Questions

### Top 10 Buildings by Consumption 

In [23]:
top_buildings = (
    df.groupby('building_id')
      ['meter_reading']
      .sum()
      .sort_values(
          ascending=False
      )
      .head(10)
)

print(top_buildings)

building_id
1108    7978.50
1072    5636.72
7       5571.77
1100    5531.25
1331    4971.84
1168    3844.93
792     3516.21
1321    3294.32
1018    3193.91
1258    3180.45
Name: meter_reading, dtype: float64


### Consumption by building type 

In [24]:
building_type = (
    df.groupby('primary_use')
      ['meter_reading']
      .sum()
      .sort_values(
          ascending=False
      )
)

print(building_type)

primary_use
Education                        64303.9065
Office                           24907.6741
Lodging/residential              15841.7741
Entertainment/public assembly    12925.1204
Public services                  11134.2950
Parking                           8001.2360
Healthcare                        6889.3297
Food sales and service            2057.9000
Utility                            600.4125
Manufacturing/industrial           537.5050
Other                              114.4167
Services                            62.3600
Warehouse/storage                   11.8333
Retail                               8.1325
Name: meter_reading, dtype: float64


### Monthly Consumption

In [25]:
monthly = (
    df.groupby('month')
      ['meter_reading']
      .sum()
)

print(monthly)

month
1     19996.0196
2     10601.7534
3     12870.6774
4      6451.3919
5     12291.0350
6     14409.9673
7     13981.8715
8     26447.7732
9      4692.2703
10     6737.5915
11     4217.9374
12    14697.6073
Name: meter_reading, dtype: float64


In [26]:
df.to_csv("../reports/cleaned_data.csv", index=False)

##